# 05 -- Contextual Thompson Sampling

**Author**: Tamer Atesyakar

We build a Bayesian logistic-regression contextual bandit from scratch, derive the Laplace approximation of its posterior via Newton-Raphson MAP fitting, and run a simulated regret study against Uniform and epsilon-Greedy baselines on a toy tone-choice problem.

**Citations**
- Russo, Van Roy, Kazerouni, Osband & Wen (2018). *A Tutorial on Thompson Sampling.* Foundations and Trends in ML.
- Chapelle & Li (2011). *An Empirical Evaluation of Thompson Sampling.* NeurIPS.
- MacKay (1992). *The Evidence Framework Applied to Classification Networks.* Neural Computation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
C_COLD = '#0f3460'
C_HOT  = '#e94560'
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

## 5.1 Problem setup

- Context: `x in R^d` summarises the user state (cognitive load, warmth, etc.).
- Arm: one of `K` tone choices (concise / encouraging / technical).
- Reward: `r in {0, 1}` (user accepted vs dismissed the adaptation).
- Model: each arm k has its own weight vector `w_k`; `p(r=1 | x, k) = sigma(x^T w_k)`.

This is one independent Bayesian logistic regression per arm.

In [ ]:
D, K = 6, 3
rng = np.random.default_rng(11)
# Ground-truth weight per arm (unknown to the learner)
W_true = rng.normal(0, 1.0, size=(K, D)) * 0.8

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def draw_context():
    return rng.normal(0, 1.0, size=D)

def pull(arm, x):
    p = sigmoid(x @ W_true[arm])
    return int(rng.random() < p), float(p)

## 5.2 Laplace approximation via Newton-Raphson MAP

We place a Gaussian prior `w ~ N(0, (1/lambda) I)` on each arm's weight. The MAP objective is the log posterior

$$\mathcal{L}(w) = \sum_t \big[ y_t \log\sigma(x_t^\top w) + (1-y_t)\log(1-\sigma(x_t^\top w)) \big] - \tfrac{\lambda}{2}\|w\|^2.$$

The Hessian at the MAP `w*` equals `-X^T S X - lambda I` where `S = diag(p(1-p))`. The Laplace posterior is then `N(w*, (-H)^{-1})` (MacKay 1992).

In [ ]:
class BayesLogReg:
    def __init__(self, d, lam=1.0):
        self.d = d; self.lam = lam
        self.mu    = np.zeros(d)
        self.Sigma = np.eye(d) / lam
        self.X = np.zeros((0, d)); self.y = np.zeros(0)

    def observe(self, x, y):
        self.X = np.vstack([self.X, x[None]])
        self.y = np.append(self.y, y)

    def fit(self, n_iter=20, tol=1e-6):
        # Newton-Raphson on the negative log posterior.
        w = self.mu.copy()
        for _ in range(n_iter):
            if self.X.shape[0] == 0:
                break
            p = sigmoid(self.X @ w)
            S = p * (1 - p)
            grad = self.X.T @ (p - self.y) + self.lam * w
            H = (self.X.T * S) @ self.X + self.lam * np.eye(self.d)
            step = np.linalg.solve(H, grad)
            w_new = w - step
            if np.linalg.norm(step) < tol:
                w = w_new; break
            w = w_new
        self.mu = w
        if self.X.shape[0] > 0:
            p = sigmoid(self.X @ w)
            S = p * (1 - p)
            H = (self.X.T * S) @ self.X + self.lam * np.eye(self.d)
            self.Sigma = np.linalg.inv(H)
        else:
            self.Sigma = np.eye(self.d) / self.lam

    def sample_weight(self):
        # One posterior sample via Cholesky.
        L = np.linalg.cholesky(self.Sigma + 1e-8 * np.eye(self.d))
        return self.mu + L @ rng.normal(size=self.d)

## 5.3 Thompson-sampling policy and baselines

In [ ]:
class Thompson:
    def __init__(self, k, d, lam=1.0, refit_every=5):
        self.models = [BayesLogReg(d, lam) for _ in range(k)]
        self.t = 0; self.refit_every = refit_every
    def choose(self, x):
        scores = [x @ m.sample_weight() for m in self.models]
        return int(np.argmax(scores))
    def update(self, a, x, y):
        self.models[a].observe(x, y)
        self.t += 1
        if self.t % self.refit_every == 0:
            for m in self.models:
                m.fit()

class EpsGreedy:
    def __init__(self, k, d, eps=0.1, lam=1.0, refit_every=5):
        self.models = [BayesLogReg(d, lam) for _ in range(k)]
        self.eps = eps; self.t = 0; self.refit_every = refit_every
    def choose(self, x):
        if rng.random() < self.eps:
            return int(rng.integers(0, len(self.models)))
        scores = [x @ m.mu for m in self.models]
        return int(np.argmax(scores))
    def update(self, a, x, y):
        self.models[a].observe(x, y)
        self.t += 1
        if self.t % self.refit_every == 0:
            for m in self.models:
                m.fit()

class Uniform:
    def __init__(self, k, d): self.k = k
    def choose(self, x): return int(rng.integers(0, self.k))
    def update(self, a, x, y): pass

## 5.4 Simulated regret

In [ ]:
def run(policy, T=800):
    cum = 0.0; trace = []
    for _ in range(T):
        x = draw_context()
        # Best possible expected reward under full knowledge of W_true:
        best = max(sigmoid(x @ W_true[k]) for k in range(K))
        a = policy.choose(x)
        r, p = pull(a, x)
        policy.update(a, x, r)
        cum += best - p
        trace.append(cum)
    return np.array(trace)

T = 800
ts  = run(Thompson(K, D))
eg  = run(EpsGreedy(K, D, eps=0.1))
un  = run(Uniform(K, D), T=T)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(un, color='#888', lw=1.2, label='Uniform')
ax.plot(eg, color=C_COLD, lw=1.8, label='eps-Greedy (0.1)')
ax.plot(ts, color=C_HOT,  lw=1.8, label='Thompson sampling')
ax.set_xlabel('t'); ax.set_ylabel('cumulative regret')
ax.set_title('Contextual bandit regret (K=3 arms, d=6)')
ax.legend()
plt.tight_layout(); plt.show()
print(f'final regret: TS={ts[-1]:.2f}, eps-G={eg[-1]:.2f}, Uniform={un[-1]:.2f}')

## 5.5 Posterior uncertainty shrinks with data

The central aesthetic of TS: early on, the posterior on `w` is wide, so samples cover a broad range of policies (exploration). As data accumulates, the posterior concentrates, and sampling converges to the MAP (exploitation). Chapelle & Li (2011) showed this achieves near-optimal empirical regret on many contextual problems.

In [ ]:
ts2 = Thompson(K, D)
trace_std = []
for t in range(400):
    x = draw_context()
    a = ts2.choose(x)
    r, _ = pull(a, x)
    ts2.update(a, x, r)
    trace_std.append(np.mean([np.sqrt(np.diag(m.Sigma)).mean() for m in ts2.models]))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(trace_std, color=C_HOT)
ax.set_title('Average posterior std across arms -- decreases with t')
ax.set_xlabel('t'); ax.set_ylabel('mean std')
plt.tight_layout(); plt.show()

## 5.6 Why Thompson sampling for I3

- It is *context-aware*: different user states activate different arms.
- The Laplace approximation is cheap (O(d^3) per refit; d is small) and on-device.
- It is *naturally exploratory* early on, a safety plus: we don't lock in a bad adaptation policy during cold-start.
- It produces calibrated uncertainty we can surface on the dashboard.